In [ ]:
peak_hour_yearly = spark.sql("""
with stats as (
    select 
        transformer_id,
        hour(hour_ts) as peak_hour,
        count(*) as days_with_increase,
        round(avg(percent_change), 2) as avg_impact
    from `electric_analysis_dev`.`transformer_hourly_analysis`
    where load_change = 'Increase' 
    group by 1, 2
),
ranked as (
    select 
        *,
        row_number() over (
            partition by transformer_id 
            order by days_with_increase desc, avg_impact desc
        ) as rnk
    from stats
)
select 
    transformer_id,
    peak_hour,
    days_with_increase,
    avg_impact
from ranked
where rnk = 1
order by avg_impact desc
""")


spark.sql("create database if not exists `electric_analysis_dev`")

peak_hour_yearly.coalesce(1).write \
    .mode("overwrite") \
    .format("parquet") \
    .option("path", "s3://ops-autopilot-data/transformed/peak_hour_yearly/") \
    .saveAsTable("`electric_analysis_dev`.`peak_hour_yearly`")